# 기본 매장별 수요 예측 학습

팀 공통 기준 모델입니다. 정기발주·재고·폐기 시뮬레이션은 포함하지 않습니다.

1~6월 학습 → 7월 모델 선택 → 7월까지 재학습 → 8~9월 평가. 최근 수요 입력은 예측일 전일까지 매일 갱신합니다. 기간별 수요가 동일하다고 검증한 것은 아니며, 월별 반복 평가가 후속 과제입니다.

실행 전 README의 환경 설치와 원본 CSV 경로를 확인하세요. 원본 데이터는 별도 제공 자료를 사용합니다. GitHub 공유본은 실행 출력이 비어 있습니다.

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'outputs'/'baseline_forecast'/'train.py').is_file())
PACKAGE=ROOT/'outputs'/'baseline_forecast'
sys.path.insert(0,str(PACKAGE))
from train import Dataset,train_forecasts
DATA_DIR=None  # 자동: data/ → data/raw/. 필요하면 Path('원본 CSV 폴더') 지정
OUTPUT_DIR=PACKAGE/'artifacts'
data=Dataset(DATA_DIR)
print('입력:',data.data_dir)


## 학습·선택·평가

mean7(최근7일), weekday4(최근4번 같은 요일), blend(50:50), hgb_own(자체 과거 피처), hgb_cross(전체 메뉴 과거 피처 추가)를 매장별 비교합니다. 선택은7월 h=0~3 WAPE만 이용합니다. h=0은 영업 전 오늘 예측이며 총14일을 예측합니다. 전체 계산은 train.py에서 확인할 수 있습니다.

In [ ]:
caches,selected,scores=train_forecasts(data,OUTPUT_DIR)
data.check_hashes()
selected

In [ ]:
scores.query("phase=='validation' and horizons=='days0to3'").pivot(index='store_id',columns='model',values='wape_pct')

In [ ]:
scores.query("phase=='test' and horizons=='day0'").pivot(index='store_id',columns='model',values='wape_pct')

## 팀원별 개선 시 지킬 비교 기준

- 기본 예측 선택은 S01 blend, S02/S03 mean7입니다. 후반 결과를 보고 기본 모델을 바꾸지 않습니다.
- 수요 목표는 demand_qty이며 메뉴 취급/행사 계획이 사전에 알려졌다고 가정합니다. 미래 실제 수요는 입력에 넣지 않습니다.
- 시간순 평가를 유지하고, 개선한 모델은 매장별·월별로 비교합니다. 전체 기간을 이미 탐색했으므로 새 미관측 성능으로 표현하지 않습니다.
- artifacts/에는 모델12개, 예측/점수/선택/명세가 저장됩니다. 생성 결과와 실행 출력이 포함된 노트북을 그대로 공개하지 마세요.
- 이 노트북은 수요 예측만 평가합니다. 예측 오차 감소만으로 부족·폐기 감소를 주장하지 않습니다.